In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve


In [14]:
df = pd.read_csv("../data/raw_drinking_water_potability.csv")

In [15]:
X = df.drop(columns="Potability")
y = df["Potability"]

y.value_counts(normalize=True)


Potability
0    0.60989
1    0.39011
Name: proportion, dtype: float64

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [17]:
numeric_features = X.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ]
)



In [31]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=5000,
        max_depth=12,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])



In [32]:
model.fit(X_train, y_train)



,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.69      0.79      0.74       400
           1       0.58      0.45      0.51       256

    accuracy                           0.66       656
   macro avg       0.63      0.62      0.62       656
weighted avg       0.65      0.66      0.65       656

ROC AUC: 0.665947265625


In [34]:
for t in [0.5, 0.45, 0.4, 0.35, 0.3]:
    y_pred_t = (y_proba >= t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred_t))



Threshold: 0.5
              precision    recall  f1-score   support

           0       0.69      0.79      0.74       400
           1       0.58      0.45      0.51       256

    accuracy                           0.66       656
   macro avg       0.63      0.62      0.62       656
weighted avg       0.65      0.66      0.65       656


Threshold: 0.45
              precision    recall  f1-score   support

           0       0.70      0.55      0.62       400
           1       0.48      0.64      0.54       256

    accuracy                           0.58       656
   macro avg       0.59      0.59      0.58       656
weighted avg       0.61      0.58      0.59       656


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.78      0.29      0.43       400
           1       0.44      0.87      0.59       256

    accuracy                           0.52       656
   macro avg       0.61      0.58      0.51       656
weighted avg       0.65  

In [35]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

idx = np.where(recall >= 0.7)[0]

best_threshold = thresholds[idx[-1]]
best_threshold



np.float64(0.43298623571030476)

In [36]:
y_pred_opt = (y_proba >= best_threshold).astype(int)

print("Threshold óptimo:", best_threshold)
print(classification_report(y_test, y_pred_opt))


Threshold óptimo: 0.43298623571030476
              precision    recall  f1-score   support

           0       0.70      0.44      0.54       400
           1       0.45      0.70      0.55       256

    accuracy                           0.54       656
   macro avg       0.57      0.57      0.54       656
weighted avg       0.60      0.54      0.54       656

